# Task 2 — Build the Portfolio Cost Hamiltonian

**Goal:** convert the portfolio objective from Task 1 into a Qiskit `SparsePauliOp` and verify that its computational-basis energies match the classical objective for all 64 bitstrings.

We use the challenge objective:

\[
C(x)=q x^T\Sigma x-\mu^T x+\lambda\left(\sum_i x_i-K\right)^2,
\qquad x_i=\frac{1-Z_i}{2}.
\]

This notebook builds and validates the Hamiltonian only. We will implement QAOA in a later task.

## 1. Imports and reproduce the challenge data

Use the same fixed random seed and parameters as Task 1 so the results are reproducible.

In [2]:
import numpy as np
from itertools import product
from qiskit.quantum_info import SparsePauliOp

rng = np.random.default_rng(2026)
n, K, q = 6, 3, 0.5
lam = 1.0

mu = rng.uniform(0.05, 0.25, n)
A = rng.normal(0, 0.1, (n, n))
sigma = A @ A.T

print("Number of stocks:", n)
print("Stocks to select:", K)
print("Penalty strength lambda:", lam)
print("Expected returns:", np.round(mu, 4))
print("Covariance matrix shape:", sigma.shape)

Number of stocks: 6
Stocks to select: 3
Penalty strength lambda: 1.0
Expected returns: [0.0858 0.178  0.1435 0.1241 0.121  0.2081]
Covariance matrix shape: (6, 6)


## 2. Define the classical objective

The function below is the reference implementation. We will compare the Hamiltonian's diagonal energy against this function for every bitstring.

In [3]:
def portfolio_cost(x, lam=lam):
    x = np.asarray(x, dtype=int)
    return float(q * (x @ sigma @ x) - mu @ x + lam * (x.sum() - K) ** 2)

print("Example cost for 011001:", round(portfolio_cost([0, 1, 1, 0, 0, 1]), 6))

Example cost for 011001: -0.503015


## 3. Expand the objective into binary linear and pairwise terms

Since \(\Sigma=A A^T\) is symmetric,

\[
q x^T\Sigma x-\mu^T x+\lambda(\sum_i x_i-K)^2
=\lambda K^2+\sum_i a_i x_i+\sum_{i<j}b_{ij}x_ix_j,
\]

where

\[
a_i=q\Sigma_{ii}-\mu_i+\lambda(1-2K),\qquad
b_{ij}=2q\Sigma_{ij}+2\lambda.
\]

Then substitute \(x_i=(I-Z_i)/2\):

- \(a_i x_i = (a_i/2)I-(a_i/2)Z_i\)
- \(b_{ij}x_ix_j=(b_{ij}/4)(I-Z_i-Z_j+Z_iZ_j)\)

The identity term is retained so the measured energy matches the full objective, including its constant offset.

In [4]:
a = q * np.diag(sigma) - mu + lam * (1 - 2 * K)
b = {}
for i in range(n):
    for j in range(i + 1, n):
        b[(i, j)] = 2 * q * sigma[i, j] + 2 * lam

print("Linear coefficients a_i:")
print(np.round(a, 6))
print("\nFirst few pair coefficients b_ij:")
for pair, coeff in list(b.items())[:5]:
    print(pair, round(float(coeff), 6))

Linear coefficients a_i:
[-5.080308 -5.173593 -5.130248 -5.086045 -5.088799 -5.198404]

First few pair coefficients b_ij:
(0, 1) 2.000811
(0, 2) 1.993587
(0, 3) 1.994449
(0, 4) 2.000099
(0, 5) 1.995133


## 4. Build the Pauli operator

Qiskit Pauli labels are ordered from qubit \(n-1\) on the left to qubit 0 on the right. The helper below places a `Z` on logical stock/qubit `i` at string position `n - 1 - i`.

We combine repeated Pauli terms before creating the `SparsePauliOp`.

In [5]:
pauli_terms = {}

def add_term(label, coeff):
    pauli_terms[label] = pauli_terms.get(label, 0.0) + float(coeff)

identity = "I" * n
add_term(identity, lam * K**2)

# Linear terms a_i x_i = a_i/2 I - a_i/2 Z_i
for i in range(n):
    z_label = ["I"] * n
    z_label[n - 1 - i] = "Z"
    add_term(identity, a[i] / 2)
    add_term("".join(z_label), -a[i] / 2)

# Pair terms b_ij x_i x_j = b_ij/4 (I - Z_i - Z_j + Z_i Z_j)
for (i, j), coeff in b.items():
    add_term(identity, coeff / 4)

    zi_label = ["I"] * n
    zi_label[n - 1 - i] = "Z"
    add_term("".join(zi_label), -coeff / 4)

    zj_label = ["I"] * n
    zj_label[n - 1 - j] = "Z"
    add_term("".join(zj_label), -coeff / 4)

    zij_label = ["I"] * n
    zij_label[n - 1 - i] = "Z"
    zij_label[n - 1 - j] = "Z"
    add_term("".join(zij_label), coeff / 4)

# Drop only numerically negligible coefficients; retain identity.
terms = [(label, coeff) for label, coeff in pauli_terms.items()
         if abs(coeff) > 1e-12]
cost_hamiltonian = SparsePauliOp.from_list(terms).simplify()

print("Hamiltonian type:", type(cost_hamiltonian).__name__)
print("Number of Pauli terms:", len(cost_hamiltonian))
print("Hamiltonian:")
print(cost_hamiltonian)

Hamiltonian type: SparsePauliOp
Number of Pauli terms: 22
Hamiltonian:
SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIZII', 'IIZIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ', 'IZIIIZ', 'ZIIIIZ', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[1.12006735+0.j, 0.04413445+0.j, 0.09052456+0.j, 0.07078699+0.j,
 0.04065603+0.j, 0.03553186+0.j, 0.09953255+0.j, 0.50020285+0.j,
 0.49839676+0.j, 0.49861213+0.j, 0.50002482+0.j, 0.49878316+0.j,
 0.49914527+0.j, 0.49863211+0.j, 0.50031986+0.j, 0.49797183+0.j,
 0.49723449+0.j, 0.49687009+0.j, 0.50269049+0.j, 0.50965842+0.j,
 0.49822937+0.j, 0.50199455+0.j])


## 5. Validate all 64 bitstrings against the classical objective

For a bitstring `bits`, character position `n-1-i` corresponds to stock/qubit `i` under Qiskit's Pauli-label convention. We calculate the diagonal energy of the operator for each computational-basis state and compare it with `portfolio_cost(x)`.

This exhaustive check is important: a sign error or reversed qubit order can produce a valid-looking Hamiltonian with incorrect answers.

In [6]:
def bitstring_to_x(bits):
    # `bits` is displayed in stock order: x[0] first, x[n-1] last.
    return np.array([int(bit) for bit in bits], dtype=int)

hamiltonian_diagonal = np.real(cost_hamiltonian.to_matrix().diagonal())

def pauli_diagonal_energy(bits):
    # `bits` is in stock order; Qiskit basis-index bit i represents qubit i.
    basis_index = sum(int(bits[i]) << i for i in range(n))
    return float(hamiltonian_diagonal[basis_index])

all_bits = ["".join(map(str, bits)) for bits in product([0, 1], repeat=n)]
max_abs_error = 0.0
validation_rows = []

for bits in all_bits:
    x = bitstring_to_x(bits)
    classical_energy = portfolio_cost(x)
    hamiltonian_energy = pauli_diagonal_energy(bits)
    error = abs(classical_energy - hamiltonian_energy)
    max_abs_error = max(max_abs_error, error)
    validation_rows.append((bits, int(x.sum()), classical_energy, hamiltonian_energy, error))

print("Bitstrings checked:", len(validation_rows))
print("Maximum absolute energy difference:", f"{max_abs_error:.3e}")
assert len(validation_rows) == 2**n
assert max_abs_error < 1e-9
print("All 64 Hamiltonian validation checks passed.")

Bitstrings checked: 64
Maximum absolute energy difference: 2.998e-15
All 64 Hamiltonian validation checks passed.


## 6. Inspect a few examples

The Hamiltonian energy should equal the direct classical cost for every shown bitstring.

In [7]:
for bits, selected_count, classical_energy, hamiltonian_energy, error in validation_rows[:8]:
    print(
        f"{bits} | selected={selected_count} | "
        f"classical={classical_energy:.6f} | "
        f"Hamiltonian={hamiltonian_energy:.6f} | "
        f"error={error:.2e}"
    )

000000 | selected=0 | classical=9.000000 | Hamiltonian=9.000000 | error=0.00e+00
000001 | selected=1 | classical=3.801596 | Hamiltonian=3.801596 | error=1.33e-15
000010 | selected=1 | classical=3.911201 | Hamiltonian=3.911201 | error=4.44e-16
000011 | selected=2 | classical=0.720775 | Hamiltonian=0.720775 | error=1.89e-15
000100 | selected=1 | classical=3.913955 | Hamiltonian=3.913955 | error=0.00e+00
000101 | selected=2 | classical=0.708468 | Hamiltonian=0.708468 | error=1.33e-15
000110 | selected=2 | classical=0.863789 | Hamiltonian=0.863789 | error=7.77e-16
000111 | selected=3 | classical=-0.333719 | Hamiltonian=-0.333719 | error=1.50e-15


## Task 2 completion checklist

- [ ] The notebook runs from top to bottom in the `qiskit-track2` kernel.
- [ ] It prints a `SparsePauliOp` Hamiltonian.
- [ ] It checks all 64 computational-basis bitstrings.
- [ ] The final output says `All 64 Hamiltonian validation checks passed.`

**Stop here after confirming the notebook works.** The next task will use this operator to construct and run QAOA; do not add QAOA code to this notebook yet.